# Гибридная система рекомендаций книг на Goodbooks-10k

Итоговый проект продолжает домашнее задание (ноутбук 01_homework_baselines.ipynb). Модели домашнего задания (item-based CF на оценках и на факте прочтения, контентная модель, SVD, популярность) становятся генераторами кандидатов. Их кандидаты объединяются в общий пул, к ним добавляются признаки книг и пар пользователь–книга, итоговый скор считается взвешенной суммой с отдельными весами для новых и активных пользователей, а финальная выдача разбавляется по жанрам. Веса подбираются на валидационной выборке, качество проверяется один раз на тестовой.

## 1. Подготовка

В Colab ячейка ниже клонирует репозиторий и ставит scikit-surprise, при локальном запуске из папки notebooks её нужно пропустить. Если scikit-surprise не собирается под numpy 2, нужно поставить numpy<2 и перезапустить среду.

In [ ]:
!git clone -q https://github.com/EgorCOStylev/goodbooks-hybrid-recsys.git
%cd goodbooks-hybrid-recsys
!pip install -q scikit-surprise

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from recsys import data, features, hybrid, metrics, pipeline
from recsys.plots import show_table

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

DATA_DIR    = Path(kagglehub.dataset_download('zygmunt/goodbooks-10k'))
SEED        = 42
THRESHOLD   = 4
K           = 10
N_PER_MODEL = 100
N_NEW       = 3000
N_TUNE      = 8000
GRID        = [-0.5, 0, 0.25, 0.5, 1, 2]
LAMBDAS     = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
SEGMENTS    = {'new': 'новые', 'active': 'активные'}

## 2. Данные

Загрузка и очистка повторяют домашнее задание: повторные пары пользователь–книга схлопываются в среднюю оценку, теги переводятся с goodreads_book_id на id из books.csv.

In [ ]:
raw       = data.load_raw(DATA_DIR)
books     = raw['books']
ratings, dup_rows, conflicts = data.dedup_ratings(raw['ratings'])
books['original_title']      = books.original_title.fillna(books.title)
book_tags                    = data.attach_tags(raw['book_tags'], books, raw['tags'])

ratings.head()

In [ ]:
print(f'Оценок после очистки: {len(ratings):,}, пользователей: {ratings.user_id.nunique():,}, книг: {ratings.book_id.nunique():,}')
print(f'Строк в повторных парах: {dup_rows:,}, из них пар с разными оценками: {conflicts:,}')

## 3. Обучающая, валидационная и тестовая выборки

Тестовая выборка совпадает с тестом домашнего задания: то же случайное разделение внутри пользователя с тем же seed, 20% оценок у каждого пользователя с не менее чем 5 оценками. Из оставшейся части тем же способом отрезается валидационная выборка. На валидации подбираются веса гибрида и параметр разнообразия, тестовая выборка используется один раз в разделе 10. В домашнем задании число соседей item-based CF выбиралось по тесту, здесь этого ограничения нет.

In [ ]:
fit, val, test = data.split_train_val_test(ratings, seed=SEED)
_, test_hw     = data.split_per_user(ratings, seed=SEED)
idx            = data.Index(users=np.sort(ratings.user_id.unique()), books=np.sort(books.id.unique()))

fit.head()

In [ ]:
for name, part in (('fit', fit), ('val', val), ('test', test)):
    print(f'{name:<5} оценок: {len(part):>8,} ({len(part) / len(ratings):.1%}), пользователей: {part.user_id.nunique():,}')
same = len(test) == len(test_hw) == len(test.merge(test_hw, on=['user_id', 'book_id']))
print(f'Тестовая выборка совпадает с тестом домашнего задания: {same}')

## 4. Сегмент новых пользователей

Пользователи, у которых меньше 5 оценок, целиком остаются в обучающей части и в оценке не участвуют, поэтому новых пользователей приходится моделировать. Из пользователей валидации, у которых в обучающей части не меньше 10 оценок, случайно выбираются 3000, и их история обрезается до 1, 2 или 3 книг, по 1000 человек на каждый вариант. Все модели обучаются уже на обрезанных данных, иначе SVD и сходства книг помнили бы удалённые оценки. Пользователь относится к сегменту новых, если у него в обучающей части не больше 3 книг.

In [ ]:
rel_val              = metrics.relevance_matrix(val, idx, THRESHOLD)
fit_t, new_val_users = pipeline.simulate_new_users(fit, rel_val, idx, n_users=N_NEW, seed=SEED)

fit_t.groupby('user_id').size().loc[new_val_users].value_counts().sort_index().to_frame('пользователей')

In [ ]:
print(f'Оценок удалено из обучающей части: {len(fit) - len(fit_t):,} ({1 - len(fit_t) / len(fit):.1%})')

## 5. Генераторы кандидатов и общий пул

Каждая модель отдаёт для пользователя 100 лучших книг, которых он не читал, и кандидаты объединяются. Генераторов шесть: item-based CF на факте прочтения (cf_implicit) и на оценках (cf_explicit), контентная модель (content), SVD, популярность по ratings_count на Goodreads (known) и неперсонализированная модель домашнего задания на сглаженной средней оценке (bayes). Популярность по ratings_count добавлена потому, что в домашнем задании модель по средней оценке оказалась на уровне случайной выдачи, а новым пользователям нужна известность книги. Параметры SVD взяты из кросс-валидации домашнего задания: 50 факторов, reg_all 0,05.

Скор каждой модели в пуле заменяется на ранг: 1 у первой книги модели, 0 у книги вне её топ-100. Так скоры с разными шкалами (прогноз оценки, косинус, сумма сходств) можно складывать.

Recall пула показывает долю релевантных книг пользователя, попавших в пул. Это потолок для любого ранжирования на втором этапе: книгу, которой нет в пуле, гибрид не порекомендует.

In [ ]:
fitted    = pipeline.fit_models(fit_t, books, book_tags, idx)
val_users = np.flatnonzero(np.diff(rel_val.indptr))
pool_val  = pipeline.scored_pool(fitted, val_users, rel_val, n_per_model=N_PER_MODEL)

segment   = hybrid.segment_of(fitted.n_ratings)
seg_users = {s: val_users[segment[val_users] == s] for s in SEGMENTS}

pool_val.head()

In [ ]:
print(f'Пользователей валидации: {len(val_users):,} (новых {len(seg_users["new"]):,}, активных {len(seg_users["active"]):,})')
print(f'Кандидатов в пуле: {len(pool_val):,}, в среднем на пользователя: {len(pool_val) / len(val_users):.0f}')

recall = pd.DataFrame({SEGMENTS[s]: hybrid.pool_recall(pool_val[pool_val.user.isin(u)], rel_val, list(fitted.generators))
                       for s, u in seg_users.items()})
recall.round(3)

In [ ]:
order = recall.sort_values('активные').index
y     = np.arange(len(order))

fig, ax = plt.subplots(figsize=(13, 5))
ax.barh(y - 0.2, recall.loc[order, 'активные'], height=0.4, color='steelblue', label='активные')
ax.barh(y + 0.2, recall.loc[order, 'новые'], height=0.4, color='coral', label='новые')
ax.set_yticks(y)
ax.set_yticklabels(order)
ax.set_xlabel(f'Recall@{N_PER_MODEL}')
ax.set_title('Доля релевантных книг, найденных каждым генератором и общим пулом (валидация)')
ax.legend()
plt.tight_layout(); plt.show()

## 6. Признаки

Признаки считаются только по обучающей части. Признаки пользователя (число оценок, средняя, разброс, доля оценок не ниже 4) одинаковы для всех его кандидатов и в линейной сумме не меняют порядок книг внутри списка, поэтому они работают через сегментацию: от числа оценок зависит, какие веса применяются. Порядок меняют признаки книги и пары: популярность по Goodreads (log_ratings_count), байесовская средняя, разброс оценок, косинус к TF-IDF-профилю пользователя (content_cos), сумма и максимум item-based сходства с прочитанным (cf_sum, cf_max), пересечение жанров книги с жанрами прочитанного (genre_affinity). Признаки пары считаются для всех кандидатов пула, в том числе для книг, которые соответствующая модель в свой топ-100 не взяла: скор CF известен и для книги, найденной контентной моделью. Каждый признак переводится в перцентиль среди кандидатов пользователя.

In [ ]:
user_feats = features.user_features(fit_t, idx)
user_feats.groupby(segment).describe().T.round(2)

In [ ]:
coverage, by_genre = features.genre_coverage(fitted.genres)
print(f'Книг хотя бы с одним жанром: {coverage:.1%}')
fitted.book_feats.describe().T.round(2)

In [ ]:
feat_cols  = list(fitted.generators) + list(pipeline.POOL_FEATURES)
separation = pool_val.groupby('hit')[feat_cols].mean().T
separation.columns = ['нерелевантные', 'релевантные']
separation['разница'] = separation['релевантные'] - separation['нерелевантные']
separation.sort_values('разница').round(3)

In [ ]:
diff = separation['разница'].sort_values()

fig, ax = plt.subplots(figsize=(13, 5))
ax.barh(diff.index, diff.values, color=np.where(diff.values > 0, 'steelblue', 'coral'))
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Средний перцентиль у релевантных минус у нерелевантных кандидатов')
ax.set_title('Насколько каждая колонка отделяет релевантные книги в пуле (валидация)')
plt.tight_layout(); plt.show()

## 7. Отдельные модели на валидации

Чтобы получить выдачу одной модели, в пуле берётся вес 1 у её колонки и 0 у остальных. Её топ-10 совпадает с топ-10 по всему каталогу, потому что пул содержит топ-100 каждой модели целиком (это проверено на синтетических данных сравнением с metrics.evaluate из домашнего задания).

In [ ]:
single = pd.DataFrame([{'модель': g, 'сегмент': SEGMENTS[s],
                        **hybrid.ranking_metrics(pool_val, hybrid.hybrid_score(pool_val, {g: 1.0}), rel_val, K, users=u)}
                       for s, u in seg_users.items() for g in fitted.generators])
single.pivot(index='модель', columns='сегмент', values=f'nDCG@{K}').sort_values('активные', ascending=False).round(4)

## 8. Подбор весов гибрида

Веса подбираются координатным подъёмом по nDCG@10: по очереди для каждой колонки перебираются значения из сетки [-0,5; 0; 0,25; 0,5; 1; 2] при остальных фиксированных, изменение принимается, если nDCG растёт, проход повторяется дважды. Отрицательный вес разрешён: колонка может штрафовать кандидата. Стартовая точка: вес 1 у item-based CF на факте прочтения, лучшей модели домашнего задания, остальные 0. Подбор идёт отдельно для новых и активных пользователей и в два этапа: сначала только ранги моделей, затем к подобранным весам моделей добавляются признаки с нулевым стартовым весом и подбор продолжается. Второй вариант стартует с решения первого, поэтому на пользователях подбора он не может оказаться хуже, а разница между ними показывает вклад признаков. При общем старте жадный поиск по большему числу колонок мог застрять в худшем локальном максимуме. Для ускорения веса активного сегмента подбираются на случайных 8000 пользователях.

In [ ]:
init_models = {g: float(g == 'cf_implicit') for g in fitted.generators}
rng         = np.random.default_rng(SEED)
tune_users  = {'new': seg_users['new'],
               'active': rng.choice(seg_users['active'], min(N_TUNE, len(seg_users['active'])), replace=False)}

weights, tuning = {}, []
for s, u in tune_users.items():
    start = hybrid.ranking_metrics(pool_val, hybrid.hybrid_score(pool_val, init_models), rel_val, K, users=u)[f'nDCG@{K}']
    for variant in ('модели', 'модели + признаки'):
        init = init_models if variant == 'модели' else {**weights[('модели', s)], **{f: 0.0 for f in pipeline.POOL_FEATURES}}
        w, best, hist = hybrid.tune_weights(pool_val, rel_val, init, GRID, k=K, users=u)
        weights[(variant, s)] = w
        tuning.append({'вариант': variant, 'сегмент': SEGMENTS[s], 'cf_implicit': start,
                       'гибрид': best, 'шагов': len(hist) - 1})

pd.DataFrame(tuning).round(4)

In [ ]:
weight_table = pd.DataFrame({f'{v}, {SEGMENTS[s]}': pd.Series(w) for (v, s), w in weights.items()}).fillna(0)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(weight_table, annot=True, fmt='.2f', cmap='RdBu_r', center=0, cbar=False, linewidths=0.5, linecolor='black', ax=ax)
ax.set_title('Подобранные веса по сегментам (0 означает, что колонка не используется)')
plt.tight_layout(); plt.show()

## 9. Разнообразие выдачи

Финальный топ-10 строится жадно методом MMR из 50 лучших кандидатов гибрида: на каждом шаге берётся книга с максимумом λ·скор − (1 − λ)·max сходства с уже выбранными, где сходство это коэффициент Жаккара между множествами жанров. При λ = 1 выдача совпадает с гибридом, с уменьшением λ в списке становится больше разных жанров. Разнообразие измеряется ILS, средним попарным сходством жанров внутри списка (меньше значит разнообразнее), и числом жанров в списке.

Правило выбора λ зафиксировано до запуска: берётся наименьшее λ, при котором nDCG@10 на валидации не ниже 98% от nDCG@10 без переранжирования.

In [ ]:
w_best    = {s: weights[('модели + признаки', s)] for s in SEGMENTS}
score_val = hybrid.segment_score(pool_val, segment, w_best)
gr_count  = books.set_index('id').ratings_count.reindex(idx.books).to_numpy()

mmr_rows = []
for lam in LAMBDAS:
    s = hybrid.mmr_rerank(pool_val, score_val, fitted.genres, k=K, lam=lam)
    mmr_rows.append({'λ': lam, **hybrid.ranking_metrics(pool_val, s, rel_val, K),
                     **hybrid.list_properties(pool_val, s, fitted.genres, gr_count, len(idx.books), K)})
mmr_table = pd.DataFrame(mmr_rows)
mmr_table.round(4)

In [ ]:
base   = mmr_table.loc[mmr_table['λ'] == 1.0, f'nDCG@{K}'].iloc[0]
LAMBDA = mmr_table.loc[mmr_table[f'nDCG@{K}'] >= 0.98 * base, 'λ'].min()
print(f'Выбранное λ: {LAMBDA}')

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(mmr_table['ILS (жанры)'], mmr_table[f'nDCG@{K}'], color='steelblue', lw=2, marker='o')
for _, row in mmr_table.iterrows():
    ax.annotate(f"λ = {row['λ']}", (row['ILS (жанры)'], row[f'nDCG@{K}']), textcoords='offset points', xytext=(6, 6))
ax.axhline(0.98 * base, color='coral', lw=2, ls='--', label='98% от nDCG без переранжирования')
ax.invert_xaxis()
ax.set_xlabel('ILS по жанрам (вправо разнообразнее)')
ax.set_ylabel(f'nDCG@{K}')
ax.set_title('Цена разнообразия на валидации')
ax.legend()
plt.tight_layout(); plt.show()

## 10. Итоговая проверка на тестовой выборке

Модели переобучаются на объединении обучающей и валидационной частей. Новые пользователи моделируются тем же способом, но среди пользователей теста. Веса и λ берутся с валидации без изменений. Сравниваются шесть отдельных моделей и три варианта гибрида: только ранги моделей, ранги моделей с признаками и последний вариант с переранжированием MMR.

In [ ]:
del pool_val

rel_test               = metrics.relevance_matrix(test, idx, THRESHOLD)
train_t, new_test_users = pipeline.simulate_new_users(pd.concat([fit, val], ignore_index=True), rel_test, idx,
                                                      n_users=N_NEW, seed=SEED)
fitted_t   = pipeline.fit_models(train_t, books, book_tags, idx)
test_users = np.flatnonzero(np.diff(rel_test.indptr))
pool_test  = pipeline.scored_pool(fitted_t, test_users, rel_test, n_per_model=N_PER_MODEL)
segment_t  = hybrid.segment_of(fitted_t.n_ratings)
seg_test   = {s: test_users[segment_t[test_users] == s] for s in SEGMENTS}

print(f'Пользователей теста: {len(test_users):,} (новых {len(seg_test["new"]):,}, активных {len(seg_test["active"]):,})')
pd.DataFrame({SEGMENTS[s]: hybrid.pool_recall(pool_test[pool_test.user.isin(u)], rel_test)
              for s, u in seg_test.items()}).round(3)

In [ ]:
systems = {g: hybrid.hybrid_score(pool_test, {g: 1.0}) for g in fitted_t.generators}
systems['гибрид: модели']            = hybrid.segment_score(pool_test, segment_t, {s: weights[('модели', s)] for s in SEGMENTS})
systems['гибрид: модели + признаки'] = hybrid.segment_score(pool_test, segment_t, w_best)
systems['гибрид + MMR']              = hybrid.mmr_rerank(pool_test, systems['гибрид: модели + признаки'], fitted_t.genres, k=K, lam=LAMBDA)

final = []
for name, score in systems.items():
    row = {'система': name}
    for s, u in (('все', test_users), *((SEGMENTS[s], u) for s, u in seg_test.items())):
        row[f'nDCG@{K}, {s}'] = hybrid.ranking_metrics(pool_test, score, rel_test, K, users=u)[f'nDCG@{K}']
    m = hybrid.ranking_metrics(pool_test, score, rel_test, K)
    row.update({f'Precision@{K}': m[f'Precision@{K}'], f'Recall@{K}': m[f'Recall@{K}'],
                **hybrid.list_properties(pool_test, score, fitted_t.genres, gr_count, len(idx.books), K)})
    final.append(row)
final = pd.DataFrame(final)
final.round(4)

In [ ]:
table = final.copy()
for col in table.columns[1:]:
    table[col] = table[col].map(lambda v: f'{v:,.0f}' if col == 'медиана популярности' else f'{v:.4f}')
show_table(table, 'Сравнение систем на тестовой выборке (медиана популярности: ratings_count на Goodreads)', fontsize=8); plt.show()

In [ ]:
plot_df = final.set_index('система')[[f'nDCG@{K}, новые', f'nDCG@{K}, активные']]
x       = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - 0.2, plot_df.iloc[:, 1], width=0.4, color='steelblue', label='активные')
ax.bar(x + 0.2, plot_df.iloc[:, 0], width=0.4, color='coral', label='новые')
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=30, ha='right')
ax.set_ylabel(f'nDCG@{K}')
ax.set_title('Отдельные модели и гибрид по сегментам (тест)')
ax.legend()
plt.tight_layout(); plt.show()

Внутри активного сегмента качество сравнивается по группам активности, как в домашнем задании: гибрид может выигрывать у пользователей с короткой историей и не давать ничего тем, у кого историй много.

In [ ]:
groups   = pd.cut(fitted_t.n_ratings[seg_test['active']], bins=[3, 10, 30, 100, 1000], labels=['4–10', '11–30', '31–100', '>100'])
by_group = pd.DataFrame({name: [hybrid.ranking_metrics(pool_test, systems[name], rel_test, K,
                                                       users=seg_test['active'][groups == g])[f'nDCG@{K}']
                                for g in groups.categories]
                         for name in ('cf_implicit', 'гибрид: модели', 'гибрид: модели + признаки')},
                        index=groups.categories)
by_group.insert(0, 'пользователей', pd.Series(groups).value_counts().reindex(groups.categories).values)
by_group.index.name = 'оценок в train'
by_group.round(4)